In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import datetime
from pathlib import Path
from os import path

from rasterio.plot import plotting_extent
from matplotlib.colors import ListedColormap
from matplotlib import colormaps
import matplotlib.ticker as mticker

import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from netCDF4 import Dataset
from wrf import getvar, get_cartopy, latlon_coords

## Formula

### Plot

In [ ]:
def plot(lats, lons, data, title, cmap='RdYlBu_r', units='', vmin=None, vmax=None,
         init_time=None, hour=None, valid_time=None):
    fig, ax = plt.subplots(figsize=(10, 8),
                            subplot_kw={'projection': ccrs.PlateCarree()})
    sc = ax.scatter(lons, lats,
                     c=data, s=0.5,
                     transform=ccrs.PlateCarree(),
                     cmap=cmap, vmin=vmin, vmax=vmax)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.add_feature(cfeature.STATES, linewidth=0.3)
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

    # marker at 57.6498, -122.1530 (Donnie Creek, BC)
    ax.scatter(-122.1530, 57.6498, s=20, color='red', marker='*',
               transform=ccrs.PlateCarree(), zorder=5)
    ax.set_extent([-109, -147, 42, 67], crs=ccrs.PlateCarree()) 

    plt.colorbar(sc, label=units, shrink=0.6)
    ax.set_title(title)
    ax.set_aspect('1.3')

    # Init / Hour / Valid annotation, similar to the GFS-style header
    if init_time is not None:
        fig.text(0.02, 0.95, f"Init: {init_time}", fontsize=11, ha='left')
    if hour is not None:
        fig.text(0.6, 0.95, f"Hour: {hour}", fontsize=11, ha='left')
    if valid_time is not None:
        fig.text(0.70, 0.95, f"Valid: {valid_time}", fontsize=11, ha='left')

    plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave room at top for the text
    plt.show()

def plot_zoom(lats, lons, data, title, cmap='RdYlBu_r', units='', vmin=None, vmax=None,
         init_time=None, hour=None, valid_time=None):
    fig, ax = plt.subplots(figsize=(10, 8),
                            subplot_kw={'projection': ccrs.PlateCarree()})
    sc = ax.scatter(lons, lats,
                     c=data, s=0.5,
                     transform=ccrs.PlateCarree(),
                     cmap=cmap, vmin=vmin, vmax=vmax)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.add_feature(cfeature.STATES, linewidth=0.3)
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

     # zoom in to the region of interest
    ax.set_extent([-112.5, -130, 52, 62.5], crs=ccrs.PlateCarree()) 
    # marker at 57.6498, -122.1530 (Donnie Creek, BC)
    ax.scatter(-122.1530, 57.6498, s=20, color='red', marker='*',
                   transform=ccrs.PlateCarree(), zorder=5)

    plt.colorbar(sc, label=units, shrink=0.6)
    ax.set_title(title)
    ax.set_aspect('1.3')

    # Init / Hour / Valid annotation, similar to the GFS-style header
    if init_time is not None:
        fig.text(0.02, 0.95, f"Init: {init_time}", fontsize=11, ha='left')
    if hour is not None:
        fig.text(0.6, 0.95, f"Hour: {hour}", fontsize=11, ha='left')
    if valid_time is not None:
        fig.text(0.70, 0.95, f"Valid: {valid_time}", fontsize=11, ha='left')

    plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave room at top for the text
    plt.show()

### Calculations

In [ ]:
# Relative Humidity Calculation Function
def calc_rh_from_t_td(t_kelvin, td_kelvin):
    """
    Calculate relative humidity (%) from 2-m temperature and 2-m dewpoint temperature (both in Kelvin), 
    using the Bolton (1980) saturation vapor pressure approximation:

        e_s(T) = 6.112 * exp( 17.67 * T_C / (T_C + 243.5) )   [hPa]
        RH     = 100 * e_s(Td) / e_s(T)
        
    where T_C is temperature in Celsius.
    """
    t_c = t_kelvin - 273.15
    td_c = td_kelvin - 273.15
    e_s_t = 6.112 * np.exp((17.67 * t_c) / (t_c + 243.5))
    e_s_td = 6.112 * np.exp((17.67 * td_c) / (td_c + 243.5))
    return 100 * (e_s_td / e_s_t)

# Files

In [ ]:
# File paths
date = datetime.datetime(2023, 5, 8) 
leadtime = 1  # valid/6

# Compute valid time as init date + forecast hour, rolling over to following days automatically
valid_hours = leadtime * 6
valid_dt = date + datetime.timedelta(hours=valid_hours)
valid_time = valid_dt.strftime('%Y-%m-%d %HZ')

wrf_dir = f"/Volumes/jchen/Share_Forecasts/WAC00WG-01/{date.strftime('%y%m%d')}00/wrfout_d02_{valid_dt.strftime('%Y-%m-%d_%H:00:00')}"
ai_bc_dir = f"/Users/jchen/Anemoi/ignition/bc/{date.strftime('%Y%m%d')}T00.nc"
ai_norway_dir = f"/Users/jchen/Anemoi/ignition/norway/{date.strftime('%Y%m%d')}T00.nc"
ai_global_dir = f"/Volumes/jchen/Share_anemoi/results/donnie_creek/global-model/{date.strftime('%Y%m%d')}T00.nc"
cx_dir = f"/Volumes/Scratch/jchen-scratch/ClimatEx/compressed-3d-wrfout/wrfout_d03_{valid_dt.strftime('%Y-%m-%d_%H:00:00')}_compressed"

# Files
wrf = xr.open_dataset(wrf_dir, engine='netcdf4')
ai_bc = xr.open_dataset(ai_bc_dir, engine='netcdf4')
ai_norway = xr.open_dataset(ai_norway_dir, engine='netcdf4')
ai_global = xr.open_dataset(ai_global_dir, engine='netcdf4')
cx = xr.open_dataset(cx_dir, engine='netcdf4')

wrf_ds = Dataset(wrf_dir)
ai_norway_ds = Dataset(ai_norway_dir)
ai_bc_ds = Dataset(ai_bc_dir)
ai_global_ds = Dataset(ai_global_dir)
cx_ds = Dataset(cx_dir)

In [ ]:
cx

### Precipitation

In [ ]:
# Anemoi BC
# leadtime
ai_bc_tds = ai_bc.sel(lead_time=ai_bc.lead_time[leadtime])

# variable
ai_bc_var = 'tp'
ai_data = ai_bc_tds[ai_bc_var].values

# coordinates
ai_bc_lat  = ai_bc_tds['latitude'].values
ai_bc_lon  = ai_bc_tds['longitude'].values

# plotting
plot(ai_bc_lat, ai_bc_lon, ai_data,
         title=f'Anemoi (BC) Total Precipitation',
         cmap='RdYlBu_r', units='?', vmin=0, vmax=2,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=valid_hours, valid_time=valid_time)

# plot_zoom(ai_bc_lat, ai_bc_lon, ai_data,
#          title=f'Anemoi (BC) Ignition Surface Pressure 2023-05-08 - Leadtime {leadtime}h',
#          cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100)

# Anemoi Norway
# leadtime
ai_norway_tds = ai_norway.sel(lead_time=ai_norway.lead_time[leadtime])

# variable
ai_norway_var = 'tp'
ai_norway_data = ai_norway_tds[ai_norway_var].values

# coordinates
ai_norway_lat  = ai_norway_tds['latitude'].values
ai_norway_lon  = ai_norway_tds['longitude'].values

# plotting
plot(ai_norway_lat, ai_norway_lon, ai_norway_data,
         title=f'Anemoi (Norway) Total Precipitation',
         cmap='RdYlBu_r', units='?', vmin=0, vmax=2,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=valid_hours, valid_time=valid_time)
# plot_zoom(ai_norway_lat, ai_norway_lon, ai_norway_data,
#          title=f'Anemoi (Norway) Ignition Surface Pressure 2023-05-08 -  Leadtime {(leadtime-1)*6}h',
#          cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100) 

# Anemoi Global
# leadtime
ai_global_tds = ai_global.sel(lead_time=ai_global.lead_time[leadtime])

# variable
ai_global_var = 'tp'
ai_global_data = ai_global_tds[ai_global_var].values

# coordinates
ai_global_lat  = ai_global_tds['latitude'].values
ai_global_lon  = ai_global_tds['longitude'].values

# plotting
plot(ai_global_lat, ai_global_lon, ai_global_data,
         title=f'Anemoi (Global) Total Precipitation',
         cmap='RdYlBu_r', units='mm', vmin=0, vmax=2,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)

# WRF
# variable
wrf_var_1 = "RAINC" 
wrf_var_2 = "RAINNC"
wrf_data = getvar(wrf_ds, wrf_var_1) + getvar(wrf_ds, wrf_var_2)


# coordinates
wrf_lats, wrf_lons = latlon_coords(getvar(wrf_ds, wrf_var_1))

# plotting
plot(wrf_lats, wrf_lons, wrf_data,
         title='WRF Ignition Surface Pressure (kPa)',
         cmap='RdYlBu_r', units='mm', vmin=0, vmax=2,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)
# plot_zoom(wrf_lats, wrf_lons, wrf_data,
#          title=f'WRF Ignition Surface Pressure 2023-05-08 - Leadtime {(leadtime-1)*6}h',
#          cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100)


### Surface Pressure

In [ ]:
# Loooooop
date_start = datetime.datetime(2023, 5, 8)
date_end   = datetime.datetime(2023, 5, 9)
save_dir = "/Users/jchen/Desktop/ignition_plots"
Path(save_dir).mkdir(parents=True, exist_ok=True)

current_date = date_start
while current_date <= date_end:
    date = current_date

    # Anemoi files depend only on the init date, so open them once per date
    ai_bc_dir = f"/Users/jchen/Anemoi/ignition/bc/{date.strftime('%Y%m%d')}T00.nc"
    ai_norway_dir = f"/Users/jchen/Anemoi/ignition/norway/{date.strftime('%Y%m%d')}T00.nc"
    ai_global_dir = f"/Volumes/jchen/Share_anemoi/results/donnie_creek/global-model/{date.strftime('%Y%m%d')}T00.nc"

    ai_bc = xr.open_dataset(ai_bc_dir, engine='netcdf4')
    ai_norway = xr.open_dataset(ai_norway_dir, engine='netcdf4')
    ai_global = xr.open_dataset(ai_global_dir, engine='netcdf4')

    ai_norway_ds = Dataset(ai_norway_dir)
    ai_bc_ds = Dataset(ai_bc_dir)
    ai_global_ds = Dataset(ai_global_dir)

    # ---- Loop over every 6-hour leadtime, 6h through 84h ----
    for valid_hours in range(6, 85, 6):
        leadtime = valid_hours // 6  # matches leadtime * 6 = valid_hours

        valid_dt = date + datetime.timedelta(hours=valid_hours)
        valid_time = valid_dt.strftime('%Y-%m-%d %HZ')

        wrf_dir = f"/Volumes/jchen/Share_Forecasts/WAC00WG-01/{date.strftime('%y%m%d')}00/wrfout_d02_{valid_dt.strftime('%Y-%m-%d_%H:00:00')}"

        try:
            wrf = xr.open_dataset(wrf_dir, engine='netcdf4')
        except Exception as e:
            print(f"  WARNING: could not open {wrf_dir} ({e}) — skipping this leadtime.")
            continue

        # Anemoi BC
        ai_bc_tds = ai_bc.sel(lead_time=ai_bc.lead_time[leadtime])
        ai_bc_var = 'sp'
        ai_data = ai_bc_tds[ai_bc_var].values / 1000
        ai_bc_lat = ai_bc_tds['latitude'].values
        ai_bc_lon = ai_bc_tds['longitude'].values

        # Anemoi Norway
        ai_norway_tds = ai_norway.sel(lead_time=ai_norway.lead_time[leadtime])
        ai_norway_var = 'sp'
        ai_norway_data = ai_norway_tds[ai_norway_var].values / 1000
        ai_norway_lat = ai_norway_tds['latitude'].values
        ai_norway_lon = ai_norway_tds['longitude'].values

        # Anemoi Global
        ai_global_tds = ai_global.sel(lead_time=ai_global.lead_time[leadtime])
        ai_global_var = 'sp'
        ai_global_data = ai_global_tds[ai_global_var].values / 1000
        ai_global_lat = ai_global_tds['latitude'].values
        ai_global_lon = ai_global_tds['longitude'].values

        # WRF
        wrf_var = "PSFC"
        wrf_ds = Dataset(wrf_dir)
        wrf_data = getvar(wrf_ds, wrf_var).values / 1000
        wrf_lats, wrf_lons = latlon_coords(getvar(wrf_ds, wrf_var))

        # ---- Plotting (unchanged) ----
        fig, axes = plt.subplots(2, 2,
                                 gridspec_kw={'wspace': 0, 'hspace': 0},
                                 figsize=(15, 13),
                                 subplot_kw={'projection': ccrs.PlateCarree()})

        datasets = [
            (ai_bc_lat, ai_bc_lon, ai_data, 'Anemoi (BC)'),
            (ai_norway_lat, ai_norway_lon, ai_norway_data, 'Anemoi (Norway)'),
            (ai_global_lat, ai_global_lon, ai_global_data, 'Anemoi (Global)'),
            (wrf_lats, wrf_lons, wrf_data, 'WRF'),
        ]

        axes_flat = axes.flatten()

        for idx, (ax, (lat, lon, data, title)) in enumerate(zip(axes_flat, datasets)):
            row, col = idx // 2, idx % 2

            sc = ax.scatter(lon, lat, c=data, s=0.5,
                             transform=ccrs.PlateCarree(),
                             cmap='RdYlBu_r', vmin=75, vmax=100)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.add_feature(cfeature.STATES, linewidth=0.3)

            gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
            gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 5))

            gl.top_labels = (row == 0)
            gl.bottom_labels = (row == 1)
            gl.left_labels = (col == 0)
            gl.right_labels = (col == 1)

            ax.scatter(-122.1530, 57.6498, s=20, color='red', marker='*',
                       transform=ccrs.PlateCarree(), zorder=5)
            ax.set_extent([-108, -148, 43, 67], crs=ccrs.PlateCarree())
            ax.set_aspect('auto')

            if row == 0:
                ax.text(0.5, 1.08, title, transform=ax.transAxes,
                        ha='center', va='bottom', fontsize=12)
            else:
                ax.text(0.5, -0.08, title, transform=ax.transAxes,
                        ha='center', va='top', fontsize=12)

        fig.text(0.10, 0.95, f"Init: {date.strftime('%Y-%m-%d')} 00Z", fontsize=14, ha='left')
        fig.text(0.70, 0.95, f"Hour: {leadtime*6}", fontsize=14, ha='left')
        fig.text(0.78, 0.95, f"Valid: {valid_time}", fontsize=14, ha='left')
        fig.suptitle('Ignition Surface Pressure', fontsize=16, y=0.99)

        cbar_ax = fig.add_axes([0.26, 0.035, 0.5, 0.02])
        cb = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal', label='kPa')
        cb.set_label('kPa', fontsize=12)
        cb.ax.tick_params(labelsize=10)

        out_path = f"{save_dir}/ignition_pressure_{date.strftime('%Y%m%d')}_leadtime{valid_hours:03d}h.png"
        plt.savefig(out_path, dpi=150)
        plt.close(fig)

        wrf.close()

    ai_bc.close()
    ai_norway.close()
    ai_global.close()

    current_date += datetime.timedelta(days=1)

print("All done!")

In [ ]:
# Anemoi BC
# leadtime
ai_bc_tds = ai_bc.sel(lead_time=ai_bc.lead_time[leadtime])

# variable
ai_bc_var = 'sp'
ai_data = ai_bc_tds[ai_bc_var].values / 1000  # Convert to kPa

# coordinates
ai_bc_lat  = ai_bc_tds['latitude'].values
ai_bc_lon  = ai_bc_tds['longitude'].values

# plotting
plot(ai_bc_lat, ai_bc_lon, ai_data,
         title=f'Anemoi (BC) Ignition Surface Pressure (kPa) - Leadtime {(leadtime-1)*6}h',
         cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)

# plot_zoom(ai_bc_lat, ai_bc_lon, ai_data,
#          title=f'Anemoi (BC) Ignition Surface Pressure 2023-05-08 - Leadtime {leadtime}h',
#          cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100)

# Anemoi Norway
# leadtime
ai_norway_tds = ai_norway.sel(lead_time=ai_norway.lead_time[leadtime])

# variable
ai_norway_var = 'sp'
ai_norway_data = ai_norway_tds[ai_norway_var].values / 1000  # Convert to kPa

# coordinates
ai_norway_lat  = ai_norway_tds['latitude'].values
ai_norway_lon  = ai_norway_tds['longitude'].values

# plotting
plot(ai_norway_lat, ai_norway_lon, ai_norway_data,
         title=f'Anemoi (Norway) Ignition Surface Pressure (kPa) - Leadtime {(leadtime-1)*6}h',
         cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)
# plot_zoom(ai_norway_lat, ai_norway_lon, ai_norway_data,
#          title=f'Anemoi (Norway) Ignition Surface Pressure 2023-05-08 -  Leadtime {(leadtime-1)*6}h',
#          cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100) 

# Anemoi Global
# leadtime
ai_global_tds = ai_global.sel(lead_time=ai_global.lead_time[leadtime])

# variable
ai_global_var = 'sp'
ai_global_data = ai_global_tds[ai_global_var].values / 1000  # Convert to kPa

# coordinates
ai_global_lat  = ai_global_tds['latitude'].values
ai_global_lon  = ai_global_tds['longitude'].values

# plotting
plot(ai_global_lat, ai_global_lon, ai_global_data,
         title=f'Anemoi (Global) Ignition Surface Pressure (kPa) - Leadtime {(leadtime-1)*6}h',
         cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)

# WRF
# variable
wrf_var = "PSFC" 
wrf_data = getvar(wrf_ds, wrf_var).values / 1000  # Convert to kPa

# coordinates
wrf_lats, wrf_lons = latlon_coords(getvar(wrf_ds, wrf_var))

# plotting
plot(wrf_lats, wrf_lons, wrf_data,
         title='WRF Ignition Surface Pressure (kPa)',
         cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)
# plot_zoom(wrf_lats, wrf_lons, wrf_data,
#          title=f'WRF Ignition Surface Pressure 2023-05-08 - Leadtime {(leadtime-1)*6}h',
#          cmap='RdYlBu_r', units='kPa', vmin=80, vmax=100)


In [ ]:
fig, axes = plt.subplots(2, 2, 
                         gridspec_kw={'wspace': 0, 'hspace': 0},
                         figsize=(15, 12),
                         subplot_kw={'projection': ccrs.PlateCarree()})

datasets = [
    (ai_bc_lat, ai_bc_lon, ai_data, 'Anemoi (BC)'),
    (ai_norway_lat, ai_norway_lon, ai_norway_data, 'Anemoi (Norway)'),
    (ai_global_lat, ai_global_lon, ai_global_data, 'Anemoi (Global)'),
    (wrf_lats, wrf_lons, wrf_data, 'WRF'),
]

axes_flat = axes.flatten()

for idx, (ax, (lat, lon, data, title)) in enumerate(zip(axes_flat, datasets)):
    row, col = idx // 2, idx % 2

    sc = ax.scatter(lon, lat, c=data, s=0.5,
                     transform=ccrs.PlateCarree(),
                     cmap='RdYlBu_r', vmin=75, vmax=100)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.add_feature(cfeature.STATES, linewidth=0.3)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 5))  # lon gridlines every 5 deg

    # Only show labels on the OUTER edges of the 2x2 block
    gl.top_labels = (row == 0)
    gl.bottom_labels = (row == 1)
    gl.left_labels = (col == 0)
    gl.right_labels = (col == 1)

    ax.scatter(-122.1530, 57.6498, s=20, color='red', marker='*',
               transform=ccrs.PlateCarree(), zorder=5)
    ax.set_extent([-108, -148, 43, 67], crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    # Top row: title above the subplot. Bottom row: title below the subplot.
    if row == 0:
        ax.text(0.5, 1.1, title, transform=ax.transAxes,
                ha='center', va='bottom', fontsize=12)
    else:
        ax.text(0.5, -0.10, title, transform=ax.transAxes,
                ha='center', va='top', fontsize=12)

fig.text(0.10, 0.95, f"Init: {date.strftime('%Y-%m-%d')} 00Z", fontsize=14, ha='left')
fig.text(0.70, 0.95, f"Hour: {(leadtime-1)*6}", fontsize=14, ha='left')
fig.text(0.78, 0.95, f"Valid: {valid_time}", fontsize=14, ha='left')
fig.suptitle('Ignition Surface Pressure', fontsize=16, y=0.99)

# Dedicated colorbar axes, placed well below the bottom row
cbar_ax = fig.add_axes([0.26, 0.02, 0.5, 0.02])
cb = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal', label='kPa')
cb.set_label('kPa', fontsize=12)
cb.ax.tick_params(labelsize=10)

plt.show()

### Relative Humidity at 2 Meters

In [ ]:
ai_bc_tds = ai_bc.sel(lead_time=ai_bc.lead_time[leadtime])
ai_norway_tds = ai_norway.sel(lead_time=ai_norway.lead_time[leadtime])
ai_global_tds = ai_global.sel(leadtime=ai_global.lead_time[leadtime])

ai_bc_var_t = '2t'
ai_bc_var_d = '2d'
ai_norway_var_t = '2t'
ai_norway_var_d = '2d'
ai_global_var_t = '2t'
ai_global_var_d = '2d'

ai_bc_data = calc_rh_from_t_td(ai_bc_tds[ai_bc_var_t].values, ai_bc_tds[ai_bc_var_d].values)
ai_norway_data = calc_rh_from_t_td(ai_norway_tds[ai_norway_var_t].values, ai_norway_tds[ai_norway_var_d].values)
ai_global_data = calc_rh_from_t_td(ai_global_tds[ai_global_var_t].values, ai_global_tds[ai_global_var_d].values)

ai_bc_lat = ai_bc_tds['latitude'].values
ai_bc_lon = ai_bc_tds['longitude'].values
ai_norway_lat = ai_norway_tds['latitude'].values
ai_norway_lon = ai_norway_tds['longitude'].values
ai_global_lat = ai_global_tds['latitude'].values
ai_global_lon = ai_global_tds['longtitude'].values

wrf_var = "rh2"
wrf_data = getvar(wrf_ds, wrf_var).values 
wrf_lats, wrf_lons = latlon_coords(getvar(wrf_ds, wrf_var))


fig, axes = plt.subplots(2, 2, 
                         gridspec_kw={'wspace': 0, 'hspace': 0},
                         figsize=(15, 12),
                         subplot_kw={'projection': ccrs.PlateCarree()})

datasets = [
    (ai_bc_lat, ai_bc_lon, ai_data, 'Anemoi (BC)'),
    (ai_norway_lat, ai_norway_lon, ai_norway_data, 'Anemoi (Norway)'),
    (ai_global_lat, ai_global_lon, ai_global_data, 'Anemoi (Global)'),
    (wrf_lats, wrf_lons, wrf_data, 'WRF'),
]

axes_flat = axes.flatten()

for idx, (ax, (lat, lon, data, title)) in enumerate(zip(axes_flat, datasets)):
    row, col = idx // 2, idx % 2

    sc = ax.scatter(lon, lat, c=data, s=0.5,
                     transform=ccrs.PlateCarree(),
                     cmap='RdYlBu_r', vmin=0, vmax=100)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.add_feature(cfeature.STATES, linewidth=0.3)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 5))  # lon gridlines every 5 deg

    # Only show labels on the OUTER edges of the 2x2 block
    gl.top_labels = (row == 0)
    gl.bottom_labels = (row == 1)
    gl.left_labels = (col == 0)
    gl.right_labels = (col == 1)

    ax.scatter(-122.1530, 57.6498, s=20, color='red', marker='*',
               transform=ccrs.PlateCarree(), zorder=5)
    ax.set_extent([-108, -148, 43, 67], crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    # Top row: title above the subplot. Bottom row: title below the subplot.
    if row == 0:
        ax.text(0.5, 1.1, title, transform=ax.transAxes,
                ha='center', va='bottom', fontsize=12)
    else:
        ax.text(0.5, -0.10, title, transform=ax.transAxes,
                ha='center', va='top', fontsize=12)

fig.text(0.10, 0.95, f"Init: {date.strftime('%Y-%m-%d')} 00Z", fontsize=14, ha='left')
fig.text(0.70, 0.95, f"Hour: {(leadtime-1)*6}", fontsize=14, ha='left')
fig.text(0.78, 0.95, f"Valid: {valid_time}", fontsize=14, ha='left')
fig.suptitle('Relative Humidity at 2 Meters', fontsize=16, y=0.99)

# Dedicated colorbar axes, placed well below the bottom row
cbar_ax = fig.add_axes([0.26, 0.02, 0.5, 0.02])
cb = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal', label='kPa')
cb.set_label('%', fontsize=12)
cb.ax.tick_params(labelsize=10)

plt.show()

In [ ]:
# Anemoi BC
# leadtime
ai_bc_tds = ai_bc.sel(lead_time=ai_bc.lead_time[leadtime])

# variable
ai_bc_var_t = '2t'
ai_bc_var_d = '2d'
ai_data = calc_rh_from_t_td(ai_bc_tds[ai_bc_var_t].values, ai_bc_tds[ai_bc_var_d].values)

# coordinates
ai_bc_lat  = ai_bc_tds['latitude'].values
ai_bc_lon  = ai_bc_tds['longitude'].values

# plotting
plot(ai_bc_lat, ai_bc_lon, ai_data,
         title=f'Anemoi (BC) Ignition Relative Humidity (%) - Leadtime {(leadtime-1)*6}h',
         cmap='BrBG', units='%', vmin=0, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)

# plot_zoom(ai_bc_lat, ai_bc_lon, ai_data,
#          title=f'Anemoi (BC) Ignition Relative Humidity (%) - Leadtime {(leadtime-1)*6}h',
#          cmap='BrBG', units='%', vmin=0, vmax=100)

# Anemoi Norway
# leadtime
ai_norway_tds = ai_norway.sel(lead_time=ai_norway.lead_time[leadtime])

# variable
ai_norway_var_t = '2t'
ai_norway_var_d = '2d'
ai_norway_data = calc_rh_from_t_td(ai_norway_tds[ai_norway_var_t].values, ai_norway_tds[ai_norway_var_d].values)

# coordinates
ai_norway_lat  = ai_norway_tds['latitude'].values
ai_norway_lon  = ai_norway_tds['longitude'].values

# plotting
plot(ai_norway_lat, ai_norway_lon, ai_norway_data,
         title=f'Anemoi (Norway) Ignition Relative Humidity (%) - Leadtime {(leadtime-1)*6}h',
         cmap='BrBG', units='%', vmin=0, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)
# plot_zoom(ai_norway_lat, ai_norway_lon, ai_norway_data,
#          title=f'Anemoi (Norway) Ignition Relative Humidity (%) -  Leadtime {(leadtime-1)*6}h',
#          cmap='BrBG', units='%', vmin=0, vmax=100) 

# WRF
# variable
wrf_var = "rh2"  # Relative Humidity at 2 meters
wrf_data = getvar(wrf_ds, wrf_var).values 

# coordinates
wrf_lats, wrf_lons = latlon_coords(getvar(wrf_ds, wrf_var))

# plotting
plot(wrf_lats, wrf_lons, wrf_data,
         title=f'WRF Ignition Relative Humidity (%) - Leadtime {(leadtime-1)*6}h',
         cmap='BrBG', units='%', vmin=0, vmax=100,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=(leadtime-1)*6, valid_time=valid_time)
# plot_zoom(wrf_lats, wrf_lons, wrf_data,
#          title=f'WRF Ignition Relative Humidity (%) - Leadtime {(leadtime-1)*6}h',
#          cmap='BrBG', units='%', vmin=0, vmax=100)


In [ ]:
xr.open_dataset("/Users/jchen/Anemoi/ignition/rh2/Anemoi_Norway.nc")

### Temperature at 2m

In [ ]:
# ClimatEX
# variable
cx_var = "T2" 
cx_data = getvar(cx_ds, cx_var).values - 273.15  # Convert to °C

# coordinates
cx_lats, cx_lons = latlon_coords(getvar(cx_ds, cx_var))

# plotting
plot(cx_lats, cx_lons, cx_data,
         title=f'ClimatEx Temperature at 2m',
         cmap='RdYlBu_r', units='°C', vmin=0, vmax=35,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=valid_hours, valid_time=valid_time)

# Anemoi BC
# leadtime
ai_bc_tds = ai_bc.sel(lead_time=ai_bc.lead_time[leadtime])

# variable
ai_bc_var = '2t'
ai_data = ai_bc_tds[ai_bc_var].values - 273.15  # Convert to °C 

# coordinates
ai_bc_lat  = ai_bc_tds['latitude'].values
ai_bc_lon  = ai_bc_tds['longitude'].values

plot(ai_bc_lat, ai_bc_lon, ai_data,
         title=f'Anemoi (BC) Temperature at 2m',
         cmap='RdYlBu_r', units='°C', vmin=0, vmax=35,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=valid_hours, valid_time=valid_time)

# Anemoi Norway
# leadtime
ai_norway_tds = ai_norway.sel(lead_time=ai_norway.lead_time[leadtime])

# variable
ai_norway_var = '2t'
ai_norway_data = ai_norway_tds[ai_norway_var].values - 273.15  # Convert to °C

# coordinates
ai_norway_lat  = ai_norway_tds['latitude'].values
ai_norway_lon  = ai_norway_tds['longitude'].values

# plotting
plot_zoom(ai_norway_lat, ai_norway_lon, ai_norway_data,
         title=f'Anemoi (Norway) Temperature at 2m',
         cmap='RdYlBu_r', units='°C', vmin=0, vmax=35,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=valid_hours, valid_time=valid_time)

# WRF
# variable
wrf_var = "T2" 
wrf_data = getvar(wrf_ds, wrf_var).values - 273.15  # Convert to °C

# coordinates
wrf_lats, wrf_lons = latlon_coords(getvar(wrf_ds, wrf_var))

# plotting
plot_zoom(wrf_lats, wrf_lons, wrf_data,
         title=f'WRF Temperature at 2m',
         cmap='RdYlBu_r', units='°C', vmin=0, vmax=35,
         init_time=f"{date.strftime('%Y-%m-%d')} 00Z", hour=valid_hours, valid_time=valid_time)
